# Monkey Thinking

Finished notebook version of the app. See `solution.py` / `main.py` in this folder for the same thing as plain scripts.

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Load meme/staring.png
meme_image = cv2.imread("../meme/staring.png")
if meme_image is None:
    print("Error: Could not load meme image")
    exit()

# Setup hand detection
base_options = python.BaseOptions(model_asset_path='../models/hand_landmarker.task')

options = vision.HandLandmarkerOptions(
    base_options = base_options,
    num_hands = 2,
    running_mode = vision.RunningMode.VIDEO,
    min_hand_detection_confidence = 0.6,
    min_tracking_confidence = 0.6)

landmarker = vision.HandLandmarker.create_from_options(options)

In [ ]:
# Open camera (tries a few indexes in case index 0 isn't your webcam)
def open_camera(max_index=3):
    for index in range(max_index):
        cap = cv2.VideoCapture(index)
        if cap.isOpened():
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            print(f"Camera opened at index {index} ({width}x{height})")
            return cap
        cap.release()
    return None

cap = open_camera()
timestamp = 0

# Check if camera opened
if cap is None:
    print("Error: Could not open camera (tried indexes 0-2)")
    exit()

print("Press 'q' to quit")

In [ ]:
# Main loop
while cap.isOpened():
    # Capture a frame and detect hands
    valid, frame = cap.read()

    if not valid:
        print("Warning: Could not read frame")
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(
        image_format = mp.ImageFormat.SRGB,
        data = rgb_frame
    )

    result = landmarker.detect_for_video(mp_image, timestamp)
    timestamp += 1

    # Check for pointing gesture and draw landmarks
    if result.hand_landmarks:
        for hand in result.hand_landmarks:
            index_tip = hand[8]
            index_pip = hand[6]
            middle_tip = hand[12]
            middle_pip = hand[10]
            ring_tip = hand[16]
            ring_pip = hand[14]
            pinky_tip = hand[20]
            pinky_pip = hand[18]

            index_extended = index_tip.y < index_pip.y
            middle_folded = middle_tip.y > middle_pip.y
            ring_folded = ring_tip.y > ring_pip.y
            pinky_folded = pinky_tip.y > pinky_pip.y

            if index_extended and middle_folded and ring_folded and pinky_folded:
                meme_image = cv2.imread("../meme/pointing.png")
            else:
                meme_image = cv2.imread("../meme/staring.png")

            for lm in hand:
                h, w, _ = frame.shape
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)
    else:
        meme_image = cv2.imread("../meme/staring.png")

    # Combine meme and camera feed, then display
    frame_height, frame_width = frame.shape[:2]
    meme_resized = cv2.resize(meme_image, (frame_width, frame_height))
    combined = np.hstack([meme_resized, frame])

    cv2.imshow('Think Monke', combined)

    if cv2.waitKey(5) & 0xFF == ord('q'):
        break

# Cleanup
cap.release()
cv2.destroyAllWindows()
landmarker.close()
print("Application closed!")